In [2]:
import ibl_info.gpfa_trajectories as gpfa
import ibl_info.utils as utils

In [3]:
# get a session with GRN
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from iblatlas.atlas import BrainRegions
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from brainbox.task.trials import get_event_aligned_raster, get_psth
from iblatlas.atlas import AllenAtlas
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from brainbox.population.decode import get_spike_counts_in_bins
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox import singlecell
from tqdm.notebook import tqdm
import seaborn as sns

from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox.singlecell import bin_spikes2D
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
df_decoding_region = pd.read_csv("../data/external/region_info.csv")
important_columns = [
    "Beryl",
    "Beryl.1",
    "# recordings",
    "# neurons",
    "# good neurons",
    "stim_dec",
    "stim_dec_sig",
    "choice_dec_sig",
    "choice_man_sig",
    "prior_dec_ep",
    "prior_dec_ep_sig",
    "prior_dec_wf_sig",
]
ONE.setup(base_url="https://openalyx.internationalbrainlab.org", silent=True)
one = ONE(password="international")
bwm_df = bwm_query(one)
unit_df = bwm_units(one)
df = df_decoding_region[important_columns]
df_with_signficant_prior = df[(df["prior_dec_ep_sig"] == True) & (df["stim_dec_sig"] == True)]
df_with_both = df_with_signficant_prior[df_with_signficant_prior["stim_dec_sig"] == True]
regions_of_interest = df_with_both["Beryl"].unique()

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01


In [6]:
grn_eids = unit_df[unit_df["Beryl"] == "GRN"]["eid"].unique()

In [7]:
sessionid = grn_eids[0]
relevant_pids = unit_df[unit_df["Beryl"] == "GRN"]["pid"].unique()
subset_df = bwm_df[bwm_df["pid"].isin(relevant_pids)]
task_list = [(row["pid"], row["eid"]) for _, row in subset_df.iterrows()]
pid, eid = task_list[-1]

In [8]:
pids, probes = one.eid2pid(eid)
if isinstance(probes, list) and len(probes) > 1:
    to_merge = [load_good_units(one, pid=pid, qc=1) for pid in pids]
    spikes, clusters = merge_probes(
        [spikes for spikes, _ in to_merge], [clusters for _, clusters in to_merge]
    )
else:
    spikes, clusters = load_good_units(one, pid=pids[0], qc=1)

In [9]:
trials, mask = load_trials_and_mask(one, eid, exclude_nochoice=False, exclude_unbiased=False)
trials = trials[mask]

In [142]:
br = BrainRegions()
region = "GRN"
acronyms = br.id2acronym(clusters["atlas_id"], mapping="Beryl")
in_region = np.isin(acronyms, [region])

if np.sum(in_region) < 10:  # Minimum unit threshold safety check
    print(f"Not enough units in {region}.")

target_ids = clusters["cluster_id"][in_region]
all_spike_ids = clusters["cluster_id"][spikes["clusters"]]
spike_mask = np.isin(all_spike_ids, target_ids)

region_spike_times = spikes["times"][spike_mask]
region_spike_ids = all_spike_ids[spike_mask]

align_event = "stimOn_times"
align_times = trials[align_event].values - 0.1  #

# binned, times = bin_spikes2D(
#     region_spike_times,
#     region_spike_ids,
#     target_ids,
#     align_times,
#     0.5,
#     0,  # type: ignore
#     0.05,
# )

binned_elephant, times = gpfa.get_spiketrains_for_elephant(
    region_spike_times,
    region_spike_ids,
    target_ids,
    align_times,
    0.5,
    0,  # type: ignore
    0.01,
)

In [150]:
mask_trials, conditions = utils.get_trial_masks(trials)

In [151]:
warnings.filterwarnings("ignore")

In [ ]:
mmd_matrix, conditions_nx, trajectories = gpfa.run_gpfa_and_mmd(
    binned_elephant, mask_trials, latent_dim=3, onlygpfa=False
)

In [160]:
accuracy = gpfa.decode_from_final_state(trajectories, trials.feedbackType)

In [10]:
a = utils.action_kernel_and_previous_feedback(sessionid, mask, True, False)

In [11]:
a[0].keys()

dict_keys(['L_Cong_Corr', 'R_Incong_Corr', 'R_Cong_Corr', 'L_Incong_Corr'])